# Figure 2: Faux Pas Follow-up and Belief Variants

This notebook generates Figure 2 following the R code logic from Supplement Analysis Figures.Rmd:
- **Panel A**: Raincloud plot comparing original "Did they know?" vs likelihood question
- **Panel B**: Horizontal bar plot showing belief likelihood variants with directional scores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 10
plt.rcParams['font.family'] = 'sans-serif'

## Define Color Palettes

In [ ]:
# Model colors (matching R code)
model_colors = {
    'GPT-3.5': '#48b5c4',
    'GPT-4': '#115f9a'
}

# Belief type colors
belief_colors = {
    'Faux Pas': '#d7658b',
    'Neutral': '#dedad2',
    'Knowledge-Implied': '#54bebe'
}

## Load Original Faux Pas Data

Following R code lines 400-403: Extract Faux Pas items from original dataset

In [ ]:
# Load the main GPT data
df_gpt = pd.read_csv('data/scores_gpt.csv')

# Filter: Faux Pas task, exclude LLaMA-70B (only GPT-3.5 and GPT-4 for panel A)
def simplify_task(task_name):
    if 'Faux Pas' in task_name:
        return 'Faux Pas'
    return task_name

df_gpt['task'] = df_gpt['task'].apply(simplify_task)
df_fp_original = df_gpt[
    (df_gpt['task'] == 'Faux Pas') & 
    (df_gpt['model'].isin(['GPT-3.5', 'GPT-4']))
].copy()

# Melt to long format (gather in R)
score_cols = [col for col in df_fp_original.columns if col.startswith('score')]
df_fp_orig_long = df_fp_original.melt(
    id_vars=['task', 'item', 'source', 'trial_state', 'model'],
    value_vars=score_cols,
    var_name='trial',
    value_name='score'
)

# Filter NA
df_fp_orig_long = df_fp_orig_long[df_fp_orig_long['score'].notna()].copy()
df_fp_orig_long = df_fp_orig_long[df_fp_orig_long['score'] != ''].copy()
df_fp_orig_long['score'] = pd.to_numeric(df_fp_orig_long['score'], errors='coerce')
df_fp_orig_long = df_fp_orig_long[df_fp_orig_long['score'].notna()].copy()

# Add question type
df_fp_orig_long['question'] = '2afc'

print(f"Original Faux Pas data: {len(df_fp_orig_long)} rows")
print(df_fp_orig_long.head())

## Load and Process Follow-up Data

Following R code lines 386-397: Load follow-up likelihood question data

In [ ]:
# Load follow-up data
df_followup_raw = pd.read_csv('data/Full_R_Project_Code/scored_data/scores_followup.csv')

# Outcome mapping (R lines 392-396)
outcome_map = {
    'A*': 'Success', 'A+': 'Success', 'A': 'Success', 'A-': 'Success', 'D': 'Success',
    'B': 'Mixed success', 'C': 'Mixed success',
    'E': 'Failure', 'F': 'Failure'
}

# Score mapping: Success=1, Mixed=0.5, Failure=0
score_map = {'Success': 1.0, 'Mixed success': 0.5, 'Failure': 0.0}

# Melt to long format
score_cols = [col for col in df_followup_raw.columns if col.startswith('score')]
df_followup_long = df_followup_raw.melt(
    id_vars=['task', 'item', 'source', 'trial_state', 'model'],
    value_vars=score_cols,
    var_name='trial',
    value_name='type'
)

# Map outcomes and scores
df_followup_long['outcome'] = df_followup_long['type'].map(outcome_map)
df_followup_long['score'] = df_followup_long['outcome'].map(score_map)

# Filter NA
df_followup_long = df_followup_long[df_followup_long['score'].notna()].copy()

# Add question type
df_followup_long['question'] = 'likely'

# Select columns to match original
df_followup_long = df_followup_long[['task', 'item', 'source', 'trial_state', 'model', 'trial', 'score', 'question']]

print(f"Follow-up data: {len(df_followup_long)} rows")
print(df_followup_long.head())

## Combine Original and Follow-up Data

Following R line 406-407

In [ ]:
# Combine both datasets
df_combined = pd.concat([df_fp_orig_long, df_followup_long], ignore_index=True)

print(f"Combined data: {len(df_combined)} rows")
print(f"Question types: {df_combined['question'].unique()}")
print(f"Models: {df_combined['model'].unique()}")

## Figure 2A: Raincloud Plot

Following R lines 414-426: Side-by-side raincloud comparison

In [ ]:
# Group by item, model, question to get mean scores (R line 415)
df_plot = df_combined.groupby(['item', 'model', 'question'], as_index=False)['score'].mean()
df_plot['point_id'] = df_plot['item'].astype(str) + '_' + df_plot['model']

print("Data for plotting:")
print(df_plot.head(20))
print(f"\nShapes by question:")
print(df_plot.groupby('question').size())

In [ ]:
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(7, 5))

# X positions
x_positions = {'2afc': 0, 'likely': 1}
x_labels = ['Did they know...?', 'Is it more likely\nthat they knew\nor didn\'t know...?']

# Plot for each model
models = ['GPT-3.5', 'GPT-4']

for model in models:
    model_data = df_plot[df_plot['model'] == model]
    
    # Draw connecting lines between questions for each item
    for item_id in model_data['item'].unique():
        item_data = model_data[model_data['item'] == item_id]
        if len(item_data) == 2:  # Has both questions
            q1_score = item_data[item_data['question'] == '2afc']['score'].values
            q2_score = item_data[item_data['question'] == 'likely']['score'].values
            
            if len(q1_score) > 0 and len(q2_score) > 0:
                ax.plot([0, 1], [q1_score[0], q2_score[0]], 
                       color=model_colors[model], alpha=0.6, linewidth=1)
    
    # Plot points for each question
    for question in ['2afc', 'likely']:
        q_data = model_data[model_data['question'] == question]
        if len(q_data) > 0:
            scores = q_data['score'].values
            x_pos = x_positions[question]
            
            # Add jitter
            jitter = np.random.normal(0, 0.02, size=len(scores))
            ax.scatter([x_pos + j for j in jitter], scores, 
                      color=model_colors[model], s=40, alpha=0.7, zorder=3)
            
            # Plot violin (half-eye) on appropriate side
            if len(scores) > 2:
                try:
                    kde = gaussian_kde(scores)
                    y_range = np.linspace(0, 1, 100)
                    density = kde(y_range)
                    density = density / density.max() * 0.15  # Scale width
                    
                    if question == '2afc':
                        # Left side violin
                        ax.fill_betweenx(y_range, x_pos - density, x_pos, 
                                        color=model_colors[model], alpha=0.3)
                    else:
                        # Right side violin
                        ax.fill_betweenx(y_range, x_pos, x_pos + density, 
                                        color=model_colors[model], alpha=0.3)
                except:
                    pass

# Formatting
ax.set_xticks([0, 1])
ax.set_xticklabels(x_labels)
ax.set_ylabel('Response score (fraction correct)')
ax.set_ylim(-0.05, 1.05)
ax.set_xlim(-0.3, 1.3)

# Legend
legend_elements = [mpatches.Patch(facecolor=model_colors[m], label=m) for m in models]
ax.legend(handles=legend_elements, loc='lower left')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/figure2a_faux_pas_followup.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure 2A saved")

## Load and Process Variants Data for Panel B

Following R lines 432-437 and 449-458

In [ ]:
# Load variants
df_variants = pd.read_csv('data/Full_R_Project_Code/scored_data/scores_variants.csv')

# Filter Faux Pas
df_fp_variants = df_variants[df_variants['Task'] == 'Faux Pas'].copy()

# Set Type ordering
df_fp_variants['Type'] = pd.Categorical(
    df_fp_variants['Type'],
    categories=['Faux Pas', 'Neutral', 'Knowledge-Implied'],
    ordered=True
)

# Drop Task column
df_fp_variants = df_fp_variants.drop('Task', axis=1)

# Melt to long format
score_cols = [col for col in df_fp_variants.columns if col.startswith('score')]
variants_long = df_fp_variants.melt(
    id_vars=['Story', 'Type'],
    value_vars=score_cols,
    var_name='trial',
    value_name='score'
)

# Extract model from trial name (R lines 455-457)
def extract_model(trial_str):
    parts = trial_str.split('-')[0]  # Get first part before '-'
    if parts == 'score_hum':
        return 'Human'
    elif parts == 'score_v4':
        return 'GPT-4'
    elif parts == 'score_v3' or 'score_v3' in trial_str:
        return 'GPT-3.5'
    elif parts == 'score_70B':
        return 'LLaMA2-70B'
    return None

variants_long['model'] = variants_long['trial'].apply(extract_model)

# Set model ordering
variants_long['model'] = pd.Categorical(
    variants_long['model'],
    categories=['Human', 'GPT-4', 'GPT-3.5', 'LLaMA2-70B'],
    ordered=True
)

# Convert scores to numeric, filter NA
variants_long['score'] = pd.to_numeric(variants_long['score'], errors='coerce')
variants_long = variants_long[variants_long['score'].notna()].copy()

# Remove LLM responses from human data (7 responses suspected to be LLM-generated, R line 444)
variants_long = variants_long[variants_long['score'] != 'LLM'].copy()

print(f"Variants data: {len(variants_long)} rows")
print(f"Models: {variants_long['model'].value_counts()}")
print(f"Types: {variants_long['Type'].value_counts()}")
print(f"Score range: {variants_long['score'].min()} to {variants_long['score'].max()}")

## Statistical Tests: Chi-square for Variants

Following R lines 449-490: Chi-square tests comparing Faux Pas and Knowledge-Implied to Neutral

In [ ]:
# Perform chi-square tests
chi_results = []

for model in ['Human', 'GPT-4', 'GPT-3.5', 'LLaMA2-70B']:
    model_data = variants_long[variants_long['model'] == model]
    
    # Test 1: Faux Pas vs Neutral
    test1_data = model_data[model_data['Type'].isin(['Faux Pas', 'Neutral'])]
    if len(test1_data) > 0:
        contingency1 = pd.crosstab(test1_data['Type'], test1_data['score'])
        if contingency1.shape[0] > 1 and contingency1.shape[1] > 1:
            chi2_1, p_1, dof_1, _ = chi2_contingency(contingency1)
            chi_results.append({
                'model': model,
                'contrast': 'Faux Pas',
                'chi2': chi2_1,
                'df': dof_1,
                'p': p_1
            })
    
    # Test 2: Knowledge-Implied vs Neutral
    test2_data = model_data[model_data['Type'].isin(['Knowledge-Implied', 'Neutral'])]
    if len(test2_data) > 0:
        contingency2 = pd.crosstab(test2_data['Type'], test2_data['score'])
        if contingency2.shape[0] > 1 and contingency2.shape[1] > 1:
            chi2_2, p_2, dof_2, _ = chi2_contingency(contingency2)
            chi_results.append({
                'model': model,
                'contrast': 'Knowledge-Implied',
                'chi2': chi2_2,
                'df': dof_2,
                'p': p_2
            })

chi_df = pd.DataFrame(chi_results)

# Apply Holm correction
if len(chi_df) > 0:
    _, p_adj, _, _ = multipletests(chi_df['p'], method='holm')
    chi_df['p_adj'] = p_adj
    chi_df['significance'] = chi_df['p_adj'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    )

print("\nChi-square test results:")
print(chi_df[['model', 'contrast', 'chi2', 'df', 'p', 'p_adj', 'significance']])

## Calculate Means and Confidence Intervals

Following R lines 515, 548-557: Using binomial CIs with Clopper-Pearson method

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def mean_ci_binomial(scores, ci=0.68):
    """
    Calculate mean and binomial CI using Clopper-Pearson method.
    Transforms scores from [-1, 0, 1] to [0, 0.5, 1] for binomial calculation.
    """
    if len(scores) == 0:
        return {'mean': np.nan, 'ci_low': np.nan, 'ci_high': np.nan}
    
    # Transform scores: -1 -> 0, 0 -> 0.5, 1 -> 1
    # This is equivalent to (score + 1) / 2
    transformed = (scores + 1) / 2
    
    mean_val = scores.mean()
    
    # For binomial CI: count "successes" as sum of transformed scores
    # (this treats each score as a weighted binomial trial)
    n = len(scores)
    successes = transformed.sum()
    
    # Clopper-Pearson binomial CI
    ci_low_transformed, ci_high_transformed = proportion_confint(
        successes, n, alpha=1-ci, method='beta'
    )
    
    # Transform back to [-1, 1] scale
    ci_low = ci_low_transformed * 2 - 1
    ci_high = ci_high_transformed * 2 - 1
    
    return {'mean': mean_val, 'ci_low': ci_low, 'ci_high': ci_high}

# Calculate means and CIs
summary_data = []

for model in ['Human', 'GPT-4', 'GPT-3.5', 'LLaMA2-70B']:
    for type_name in ['Faux Pas', 'Neutral', 'Knowledge-Implied']:
        subset = variants_long[
            (variants_long['model'] == model) & 
            (variants_long['Type'] == type_name)
        ]
        
        if len(subset) > 0:
            scores = subset['score'].values
            ci_data = mean_ci_binomial(scores, ci=0.68)
            
            summary_data.append({
                'model': model,
                'Type': type_name,
                'mean': ci_data['mean'],
                'ci_low': ci_data['ci_low'],
                'ci_high': ci_data['ci_high'],
                'n': len(scores)
            })

summary_df = pd.DataFrame(summary_data)

print("\nSummary statistics with 68% binomial CIs:")
print(summary_df)

## Figure 2B: Horizontal Bar Plot with Significance Markers

Following R lines 532-567

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Model order (will be reversed for horizontal bars)
models = ['Human', 'GPT-4', 'GPT-3.5', 'LLaMA2-70B']
y_positions = {model: i for i, model in enumerate(models[::-1])}  # Reverse for plotting

# Type offsets for dodging
bar_width = 0.22
type_offsets = {
    'Faux Pas': -0.3,
    'Neutral': 0.0,
    'Knowledge-Implied': 0.3
}

# Plot bars
for model in models:
    y_base = y_positions[model]
    
    for type_name in ['Faux Pas', 'Neutral', 'Knowledge-Implied']:
        subset = summary_df[
            (summary_df['model'] == model) & 
            (summary_df['Type'] == type_name)
        ]
        
        if len(subset) > 0:
            row = subset.iloc[0]
            mean_val = row['mean']
            ci_low = row['ci_low']
            ci_high = row['ci_high']
            n_samples = row['n']
            
            y = y_base + type_offsets[type_name]
            
            # Draw horizontal bar
            ax.barh(y, mean_val, height=bar_width, 
                   color=belief_colors[type_name],
                   edgecolor='black', linewidth=0.8)
            
            # Draw error bars
            ax.errorbar(mean_val, y,
                       xerr=[[mean_val - ci_low], [ci_high - mean_val]],
                       fmt='o', color='black', markersize=4,
                       capsize=3, linewidth=1.2, zorder=10)
            
            # Add frequency circles (on the right side)
            # Scale circle size based on frequency
            circle_size = n_samples * 1.5  # Adjust scaling factor
            circle_x = 1.0  # Position on right side
            
            ax.scatter(circle_x, y, s=circle_size, 
                      color=belief_colors[type_name], 
                      alpha=0.6, edgecolor='black', linewidth=0.5, zorder=5)

# Add p-values next to circles on the right
p_value_x = 1.15  # Position for p-value text

for _, row in chi_df.iterrows():
    model = row['model']
    contrast = row['contrast']
    p_adj = row['p_adj']
    
    y_base = y_positions[model]
    y_contrast = y_base + type_offsets[contrast]
    
    # Format p-value
    if p_adj < 0.001:
        # Scientific notation
        exp = int(np.floor(np.log10(p_adj)))
        mantissa = p_adj / (10 ** exp)
        p_text = f"{mantissa:.2f} × 10$^{{{exp}}}$"
    else:
        p_text = f"{p_adj:.3f}"
    
    # Color based on contrast type
    text_color = belief_colors[contrast]
    
    # Add p-value text
    ax.text(p_value_x, y_contrast, p_text,
           va='center', ha='left', fontsize=9,
           color=text_color, fontweight='bold')

# Add "P values:" label
ax.text(p_value_x, y_positions['Human'] + 0.5, 'P values:',
       va='bottom', ha='left', fontsize=9, style='italic')

# Add frequency legend in top LEFT corner
legend_frequencies = [50, 100, 150, 200, 250]
legend_x = -0.95  # Left side
legend_y_start = y_positions['Human'] - 0.4  # Start from top

ax.text(legend_x, legend_y_start + 0.5, 'Frequency',
       va='bottom', ha='center', fontsize=9, fontweight='bold')

for i, freq in enumerate(legend_frequencies):
    circle_size = freq * 1.5
    legend_y = legend_y_start + 0.35 - i * 0.15
    ax.scatter(legend_x, legend_y, s=circle_size,
              color='lightgray', alpha=0.5, edgecolor='black', linewidth=0.5)
    ax.text(legend_x + 0.08, legend_y, str(freq),
           va='center', ha='left', fontsize=8)

# Vertical line at 0
ax.axvline(0, color='grey', linestyle='--', linewidth=1, zorder=0)

# Formatting
ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels(models[::-1])
ax.set_xlim(-1.2, 1.45)
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(["Didn't know\n(-1)", "Unsure\n(0)", "Knew\n(+1)"])
ax.set_xlabel('"More likely that they..."', fontsize=11)

# Legend for bar types
legend_elements = [
    mpatches.Patch(facecolor=belief_colors[t], edgecolor='black', label=t)
    for t in ['Faux Pas', 'Neutral', 'Knowledge-Implied']
]
ax.legend(handles=legend_elements, loc='upper left', frameon=False, ncol=3,
         bbox_to_anchor=(0.0, 1.02))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/figure2b_belief_variants.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure 2B saved")

## Summary

In [ ]:
print("\n=== Figure 2 Generation Complete ===")
print("\nPanel A: Faux Pas Follow-up comparison")
print("  - Shows improvement from original to likelihood question")
print("  - Saved to: figures/figure2a_faux_pas_followup.png")
print("\nPanel B: Belief Likelihood Variants")
print("  - Shows directional biases for Faux Pas, Neutral, and Knowledge-Implied")
print("  - Saved to: figures/figure2b_belief_variants.png")
print("\nStatistical tests with Holm correction applied")